# NiriKsha — Legal Metrology Packaged-Commodity Inspection Prototype

End-to-end Colab prototype: **inspection details → front/back/side/additional images → blur gate → OCR → declaration extraction → multi-image fusion/conflict detection → deterministic rules → findings → PDF report**.

**Authoritative legal source:** the attached *The Legal Metrology (Package Commodities) Rules, 2011.pdf*. This notebook intentionally uses that supplied PDF as its legal source and does not silently substitute later amendments or outside legal sources.

**Safety:** this is decision-support, not an autonomous legal authority. OCR uncertainty becomes **NEEDS_MANUAL_VERIFICATION**, not a legal violation. Photographs cannot verify the package's actual physical net quantity; physical quantity testing is a separate inspection process described in Rule 19 and the schedules.

In [ ]:
!apt-get update -qq
!apt-get install -y tesseract-ocr -qq
!pip install -q pytesseract opencv-python-headless reportlab numpy Pillow nbformat

import re, os, json, hashlib
from pathlib import Path
from datetime import datetime
import cv2, numpy as np, pytesseract
from pytesseract import Output
import matplotlib.pyplot as plt
from google.colab import files

## 1. Inspection details and package-image input

Upload the same types of package views the mobile workflow asks for:

- **FRONT**
- **BACK**
- **SIDE**
- optional **ADDITIONAL** views

Filenames containing `front`, `back`, or `side` are automatically tagged. Other files become `additional_1`, `additional_2`, etc.

In [ ]:
def collect_inspection_input():
    return {
        "inspection_id": f"COLAB-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
        "product_name": input("Product / commodity name: ").strip(),
        "brand": input("Brand: ").strip(),
        "category": input("Category [food / household_personal_care]: ").strip().lower(),
        "manufacturer": input("Manufacturer (optional): ").strip(),
        "batch": input("Batch / lot (optional): ").strip(),
        "location": input("Inspection location (optional): ").strip(),
        "imported": input("Imported package? [y/n]: ").strip().lower() == "y",
        "captured_at": datetime.now().isoformat(timespec="seconds"),
    }

def upload_package_images():
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No images uploaded.")
    images = {}
    for filename, raw in uploaded.items():
        img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            continue
        low = filename.lower()
        if "front" in low: role = "front"
        elif "back" in low: role = "back"
        elif "side" in low: role = "side"
        else: role = f"additional_{sum(k.startswith('additional_') for k in images)+1}"
        if role in images:
            role = f"additional_{sum(k.startswith('additional_') for k in images)+1}"
        images[role] = {"filename": filename, "image": img,
                        "sha256": hashlib.sha256(raw).hexdigest()}
    if not images: raise ValueError("No valid images were decoded.")
    return images

inspection_input = collect_inspection_input()
package_images = upload_package_images()

for role, item in package_images.items():
    plt.figure(figsize=(6,4))
    plt.imshow(cv2.cvtColor(item["image"], cv2.COLOR_BGR2RGB))
    plt.title(f"{role.upper()} — {item['filename']}")
    plt.axis("off"); plt.show()

print("Inspection:", inspection_input["inspection_id"])
print("Views:", list(package_images))

## 2. Image-quality gate — Variance of Laplacian

This follows the requested OpenCV method. Images are downscaled only when wider than 800 px; they are never upscaled.

`150.0` is a **prototype calibration threshold**, not a legal threshold.

In [ ]:
BLUR_THRESHOLD = 150.0
BLUR_TARGET_WIDTH = 800

def check_blur(image, threshold=BLUR_THRESHOLD, target_width=BLUR_TARGET_WIDTH):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    if w > target_width:
        gray = cv2.resize(gray, (target_width, max(1, int(h*target_width/w))),
                          interpolation=cv2.INTER_LINEAR)
    score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    blurry = score < threshold
    return {"status":"blurry" if blurry else "acceptable", "score":score,
            "threshold":float(threshold), "action":"retake" if blurry else "proceed"}

def run_quality_gate(images):
    results, failed = {}, []
    for role, item in images.items():
        r = check_blur(item["image"])
        r.update(image_role=role, filename=item["filename"], sha256=item["sha256"])
        results[role] = r
        print(f"{role.upper():<14} score={r['score']:.2f}  "
              f"threshold={r['threshold']:.1f}  {r['status'].upper()}")
        if r["status"] == "blurry": failed.append(role)
    return results, failed

quality_results, failed_images = run_quality_gate(package_images)
if failed_images:
    print("RETAKE PHOTO BEFORE PROCEEDING:", failed_images)
else:
    print("All uploaded images passed the blur gate.")

## 3. OCR — isolated and swappable

Tesseract is the default open-source OCR engine. Preprocessing uses grayscale plus adaptive thresholding. OCR is run independently for each accepted image and retains confidence/bounding boxes.

In [ ]:
OCR_CONFIDENCE_THRESHOLD = 55.0

def preprocess_for_ocr(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3,3), 0)
    return cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 31, 11)

def extract_text(image, image_role=None):
    processed = preprocess_for_ocr(image)
    data = pytesseract.image_to_data(processed, config="--psm 6",
                                     output_type=Output.DICT)
    words, confs, boxes = [], [], []
    for i, txt in enumerate(data["text"]):
        txt = (txt or "").strip()
        try: conf = float(data["conf"][i])
        except: conf = -1
        if txt and conf >= 0:
            boxes.append({"text":txt, "confidence":conf,
                          "bbox":[int(data["left"][i]),int(data["top"][i]),
                                  int(data["width"][i]),int(data["height"][i])]})
            words.append(txt); confs.append(conf)
    return {"image_role":image_role, "raw_text":" ".join(words).strip(),
            "mean_confidence":float(np.mean(confs)) if confs else 0.0,
            "words":boxes, "status":"detected" if words else "not_found"}

def run_ocr(images, quality):
    out = {}
    for role, item in images.items():
        if quality[role]["status"] == "blurry":
            out[role] = {"image_role":role, "status":"skipped_quality_gate",
                         "raw_text":"","mean_confidence":0.0,"words":[]}
            continue
        r = extract_text(item["image"], role)
        r["filename"] = item["filename"]
        out[role] = r
        print(f"\n===== {role.upper()} OCR =====")
        print("Confidence:", round(r["mean_confidence"],2))
        print(r["raw_text"] or "[NO TEXT DETECTED]")
    return out

ocr_results = run_ocr(package_images, quality_results) if not failed_images else {
    r: {"image_role":r,"status":"blocked_by_quality_gate","raw_text":""}
    for r in package_images
}

## 4. Declaration extraction

This layer is deterministic/best-effort. It does **not** decide compliance. Each candidate keeps its source image and OCR confidence.

In [ ]:
FIELDS = ["commodity_name","manufacturer_details","net_quantity","mrp",
          "date_of_manufacture_packing_import","consumer_care_details","country_of_origin"]

def first_match(pattern, text):
    m = re.search(pattern, text, re.I|re.M)
    return m.group(1).strip() if m else None

def extract_declarations(ocr):
    text, role, conf = ocr.get("raw_text",""), ocr.get("image_role"), ocr.get("mean_confidence",0)
    out = []
    def add(field, value, status="detected"):
        if value:
            out.append({"field":field,"value":value,"status":status,
                        "confidence":conf,"source_image":role,
                        "evidence_text":text[:700]})
    add("mrp", first_match(
        r"(?:M\.?R\.?P\.?|MAX(?:IMUM)?\s+RETAIL\s+PRICE)[^\d₹RsINR]*"
        r"(?:₹|Rs\.?|INR)?\s*([0-9]+(?:\.[0-9]{1,2})?)", text))
    add("net_quantity", first_match(
        r"(?:NET\s*(?:QTY|QUANTITY|WT|WEIGHT|CONTENT))[^\d]*(\d+(?:\.\d+)?)\s*"
        r"(kg|kgs|g|gm|grams?|l|litre|litres|ml|millilitre|millilitres|m|cm|mm|N|U)?", text))
    add("date_of_manufacture_packing_import", first_match(
        r"(?:MFG|MFD|PKD|PACKED|MANUFACTURED|DATE)[^\n]{0,25}"
        r"((?:0?[1-9]|1[0-2])\s*[/\-.]\s*(?:20)?\d{2}|"
        r"(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[A-Z]*\s+\d{4})", text))
    add("country_of_origin", first_match(
        r"(?:COUNTRY\s+OF\s+ORIGIN|MADE\s+IN)\s*[:\-]?\s*([A-Za-z][A-Za-z .'-]{2,40})", text))
    phone = first_match(r"((?:\+91[-\s]?)?[6-9]\d{9}|1800[-\s]?\d{3}[-\s]?\d{3,4})", text)
    email = first_match(r"([A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,})", text)
    care = []
    if phone: care.append("phone: "+phone)
    if email: care.append("email: "+email)
    if re.search(r"(consumer|customer|care|complaint|grievance|helpline)", text, re.I):
        care.append("consumer-care wording detected")
    add("consumer_care_details", "; ".join(care))
    maker = first_match(
        r"((?:MANUFACTURED\s+BY|MANUFACTURER|PACKED\s+BY|PACKER|IMPORTED\s+BY|IMPORTER)"
        r"[^\n]{0,250})", text)
    add("manufacturer_details", maker,
        "detected" if conf >= OCR_CONFIDENCE_THRESHOLD else "low_confidence")
    return out

per_image = {role: extract_declarations(o) for role,o in ocr_results.items()
             if o.get("status") not in {"blocked_by_quality_gate","skipped_quality_gate"}}

for role, ds in per_image.items():
    print(f"\n{role.upper()}:")
    for d in ds: print(" ", d["field"], "=>", d["value"], f"(conf {d['confidence']:.1f})")

## 5. Multi-image fusion and conflict detection

The same field can appear on multiple panels. Equal values are merged; different values become a conflict and require manual verification. No conflicting value is auto-selected.

In [ ]:
def norm(v): return re.sub(r"\s+"," ",str(v).strip().lower())

def merge_declarations(per_image):
    grouped = {f:[] for f in FIELDS}
    for role, ds in per_image.items():
        for d in ds: grouped.setdefault(d["field"],[]).append(d)
    merged, conflicts = {}, []
    for field, candidates in grouped.items():
        if not candidates:
            merged[field] = {"field":field,"value":None,"status":"not_found",
                             "confidence":0.0,"source_images":[],"candidates":[]}
            continue
        vals = {}
        for c in candidates: vals.setdefault(norm(c["value"]),[]).append(c)
        if len(vals) > 1:
            conflicts.append({"field":field,"status":"conflicting",
                              "candidates":[{"value":c["value"],
                                            "source_image":c["source_image"],
                                            "confidence":c["confidence"]} for c in candidates],
                              "action":"NEEDS_MANUAL_VERIFICATION"})
            merged[field] = {"field":field,"value":None,"status":"conflicting",
                             "confidence":min(c["confidence"] for c in candidates),
                             "source_images":sorted({c["source_image"] for c in candidates}),
                             "candidates":[c["value"] for c in candidates]}
        else:
            best=max(candidates,key=lambda x:x["confidence"])
            merged[field]={"field":field,"value":best["value"],"status":best["status"],
                           "confidence":best["confidence"],
                           "source_images":sorted({c["source_image"] for c in candidates}),
                           "candidates":[c["value"] for c in candidates],
                           "evidence_text":best.get("evidence_text","")}
    return merged, conflicts

merged_declarations, conflicts = merge_declarations(per_image)
print("Conflicts:", len(conflicts))
for c in conflicts: print(c)

# 6. PDF-sourced deterministic rules

The registry is based on provisions visible in the supplied PDF, including **Rule 3, Rule 6, Rule 9(4), Rule 10, Rule 13 and Rule 19**, plus the **Rule 26** exemptions visible in the supplied PDF.

The legal engine deliberately does not attempt every provision in the 43-page document. It is a focused prototype and does not certify full legal compliance.

The supplied PDF explicitly states that Rule 13 requires SI units, gives unit conventions below/at thresholds, and prohibits `dozen`, `score`, `gross`, and `great gross`. Rule 9(4) requires the particulars to be in Hindi in Devanagari or English, with other languages permitted additionally. Rule 19 describes physical sample testing, which is outside photographic verification.

In [ ]:
LEGAL_SOURCE = "The Legal Metrology (Package Commodities) Rules, 2011.pdf — supplied PDF"
RULE_VERSION = "SUPPLIED_PDF_2011"

RULES = [
 {"rule_id":"PCR-R6-A","name":"Manufacturer / packer / importer details",
  "provision":"Rule 6(1)(a); Rule 10","field":"manufacturer_details"},
 {"rule_id":"PCR-R6-B","name":"Country of origin for imported package",
  "provision":"Rule 6(1)(b)","field":"country_of_origin","conditional":"imported"},
 {"rule_id":"PCR-R6-C","name":"Net quantity",
  "provision":"Rule 6(1)(c); Rule 13","field":"net_quantity"},
 {"rule_id":"PCR-R6-D","name":"Manufacture / pre-packing / import date",
  "provision":"Rule 6(1)(d)","field":"date_of_manufacture_packing_import"},
 {"rule_id":"PCR-R6-E","name":"Retail sale price",
  "provision":"Rule 6(1)(e); Rule 9","field":"mrp"},
 {"rule_id":"PCR-R6-F","name":"Name of commodity",
  "provision":"Rule 6(1)(f)","field":"commodity_name"},
 {"rule_id":"PCR-R6-G","name":"Consumer-care / complaint contact",
  "provision":"Rule 6(1)(g)","field":"consumer_care_details"},
 {"rule_id":"PCR-R9-LANG","name":"Declaration language",
  "provision":"Rule 9(4)","field":None,"manual_visual":True},
 {"rule_id":"PCR-R13-UNITS","name":"Net-quantity unit convention",
  "provision":"Rule 13(1)-(5)","field":"net_quantity"},
]

# Rule 26 exemptions explicitly visible in the supplied PDF.
R26 = [
 ("R26-A","Rule 26(a)","<=10 g or <=10 ml exemption; proviso retains MRP and net quantity for 10–20 g/ml."),
 ("R26-B","Rule 26(b)","Fast food items packed by restaurant/hotel and the like."),
 ("R26-C","Rule 26(c)","Scheduled/non-scheduled formulations covered under the Drugs (Price Control) Order, 1995, as stated in this supplied PDF."),
 ("R26-D","Rule 26(d)","Agricultural farm produce in packages above 50 kg."),
]

print("Rules:", len(RULES))
print("Rule 26 exemptions represented:", len(R26))

In [ ]:
def parse_quantity(value):
    if not value: return None,None
    m=re.search(r"(\d+(?:\.\d+)?)\s*(kg|kgs|g|gm|grams?|l|litre|litres|ml|millilitre|millilitres|m|cm|mm|N|U)\b",
                str(value), re.I)
    return (float(m.group(1)),m.group(2).lower()) if m else (None,None)

def quantity_check(value):
    q,u=parse_quantity(value)
    if q is None: return "WARNING","Quantity/unit could not be parsed reliably."
    if u in {"kg","kgs"} and q < 1: return "WARNING","Rule 13: below 1 kg should be expressed in grams."
    if u in {"l","litre","litres"} and q < 1: return "WARNING","Rule 13: below 1 litre should be expressed in millilitres."
    return "PASS","Unit is compatible with the supported Rule 13 check."

def data_quality_checks(decls):
    warnings=[]
    mrp=decls.get("mrp",{}).get("value")
    if mrp:
        try:
            if float(re.sub(r"[^0-9.]","",str(mrp))) < 0:
                warnings.append({"rule_id":"VAL-MRP-NONNEGATIVE","status":"WARNING","detail":"Negative MRP parsed."})
        except:
            warnings.append({"rule_id":"VAL-MRP-NUM","status":"WARNING","detail":"MRP detected but not numeric."})
    qty=decls.get("net_quantity",{}).get("value")
    if qty:
        s,d=quantity_check(qty)
        if s=="WARNING": warnings.append({"rule_id":"VAL-QTY-UNIT","status":"WARNING","detail":d})
    return warnings

def check_compliance(decls, product_context):
    findings=[]; exemptions=[]
    q,u=parse_quantity(decls.get("net_quantity",{}).get("value"))
    if q is not None and u in {"g","gm","grams"} and q <= 10:
        exemptions.append("Rule 26(a): package quantity <= 10 g")
    if q is not None and u in {"ml","millilitre","millilitres"} and q <= 10:
        exemptions.append("Rule 26(a): package quantity <= 10 ml")

    for rule in RULES:
        if rule.get("conditional")=="imported" and not product_context.get("imported"):
            findings.append({**rule,"status":"NOT_APPLICABLE",
                             "detail":"Inspection context says package is not imported."})
            continue
        if rule.get("manual_visual"):
            findings.append({**rule,"status":"NEEDS_MANUAL_VERIFICATION",
                             "detail":"Language compliance cannot be reliably certified from OCR alone."})
            continue
        if exemptions:
            findings.append({**rule,"status":"NOT_APPLICABLE",
                             "detail":"Rule 26(a) exemption applied."})
            continue

        field=rule["field"]
        value=decls.get(field,{})
        if value.get("status")=="conflicting":
            status,detail="NEEDS_MANUAL_VERIFICATION","Conflicting values across images."
        elif not value.get("value"):
            status,detail="NEEDS_MANUAL_VERIFICATION","OCR did not reliably detect the declaration; absence is not established."
        elif value.get("confidence",0) < OCR_CONFIDENCE_THRESHOLD:
            status,detail="UNREADABLE","OCR confidence is below the configured threshold."
        elif rule["rule_id"]=="PCR-R13-UNITS":
            status,detail=quantity_check(value["value"])
        else:
            status,detail="PASS","Declaration detected with sufficient OCR confidence."

        findings.append({
            **rule,"status":status,"detail":detail,
            "value":value.get("value"),"confidence":value.get("confidence"),
            "source_images":value.get("source_images",[])
        })

    warnings=data_quality_checks(decls)
    unresolved=[f for f in findings if f["status"] in
                {"NEEDS_MANUAL_VERIFICATION","UNREADABLE","POTENTIAL_NON_COMPLIANCE"}]
    overall="NEEDS_MANUAL_VERIFICATION" if unresolved else "NO_POTENTIAL_VIOLATIONS"
    return {"overall_status":overall,"legal_findings":findings,
            "data_quality_warnings":warnings,"exemptions_applied":exemptions,
            "legal_source":LEGAL_SOURCE,"rule_version":RULE_VERSION}

merged_for_rules=dict(merged_declarations)
merged_for_rules["commodity_name"]={
    "field":"commodity_name","value":inspection_input["product_name"] or None,
    "status":"manual_context" if inspection_input["product_name"] else "not_found",
    "confidence":100.0 if inspection_input["product_name"] else 0.0,
    "source_images":[],"evidence_text":"Inspection metadata, not photographic OCR evidence."
}

compliance_result=check_compliance(merged_for_rules, inspection_input)
for f in compliance_result["legal_findings"]:
    print(f"{f['rule_id']:<16} {f['status']:<28} {f['detail']}")
print("\nData-quality warnings:", compliance_result["data_quality_warnings"])
print("Exemptions:", compliance_result["exemptions_applied"])
print("Overall:", compliance_result["overall_status"])

# 7. PDF inspection report

The report contains inspection metadata, image-quality results, OCR summaries, declarations, conflicts, legal findings, data-quality warnings, exemptions, evidence images, source/provision references, and safety limitations.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import mm

LEGAL_SAFETY = ("This system provides AI-assisted preliminary inspection findings and decision support. "
" It evaluates only a focused subset of machine-verifiable requirements and does not replace the legal "
"authority of an enforcement officer. It does not independently issue legally binding determinations.")
QUANTITY_LIMIT = ("A photograph can evaluate the printed net-quantity declaration but cannot verify whether "
"the package physically contains the declared quantity. Physical quantity verification requires physical "
"measurement/weighing and the applicable inspection procedure.")

def build_result():
    return {
        "inspection":inspection_input,"package_images":package_images,
        "image_quality":quality_results,"ocr":ocr_results,
        "declarations":merged_for_rules,"conflicts":conflicts,
        "legal_findings":compliance_result["legal_findings"],
        "data_quality_warnings":compliance_result["data_quality_warnings"],
        "exemptions_applied":compliance_result["exemptions_applied"],
        "overall_status":compliance_result["overall_status"],
        "legal_source":LEGAL_SOURCE,"rule_version":RULE_VERSION,
        "disclaimer":LEGAL_SAFETY,"physical_quantity_disclaimer":QUANTITY_LIMIT
    }

def generate_report(result, output_path):
    styles=getSampleStyleSheet()
    doc=SimpleDocTemplate(output_path,pagesize=A4,leftMargin=14*mm,rightMargin=14*mm,
                          topMargin=14*mm,bottomMargin=14*mm)
    story=[Paragraph("NiriKsha",styles["Title"]),
           Paragraph("AI-Assisted Legal Metrology Inspection — Prototype Report",styles["Heading2"])]
    meta=result["inspection"]
    rows=[["Inspection ID",meta["inspection_id"]],["Product",meta["product_name"]],
          ["Brand",meta["brand"]],["Category",meta["category"]],["Location",meta["location"]],
          ["Imported",str(meta["imported"])],["Overall status",result["overall_status"]]]
    t=Table(rows,colWidths=[45*mm,135*mm]); t.setStyle(TableStyle([
        ("GRID",(0,0),(-1,-1),.4,colors.grey),("BACKGROUND",(0,0),(0,-1),colors.whitesmoke),
        ("FONTSIZE",(0,0),(-1,-1),8)]))
    story += [Paragraph("1. Inspection Information",styles["Heading2"]),t,Spacer(1,8)]

    story += [Paragraph("2. Image Quality",styles["Heading2"])]
    qrows=[["View","Score","Threshold","Status"]]+[
        [r,f"{v['score']:.2f}",f"{v['threshold']:.1f}",v["status"].upper()]
        for r,v in result["image_quality"].items()]
    qt=Table(qrows); qt.setStyle(TableStyle([("GRID",(0,0),(-1,-1),.4,colors.grey),
                                             ("BACKGROUND",(0,0),(-1,0),colors.whitesmoke),
                                             ("FONTSIZE",(0,0),(-1,-1),8)]))
    story += [qt]

    story += [Paragraph("3. OCR Summary",styles["Heading2"])]
    for role,o in result["ocr"].items():
        story += [Paragraph(f"<b>{role.upper()}</b> — {o.get('status')} — confidence {o.get('mean_confidence',0):.1f}",styles["BodyText"]),
                  Paragraph((o.get("raw_text") or "[no text detected]")[:1800].replace("&","&amp;"),styles["BodyText"]),
                  Spacer(1,4)]

    story += [Paragraph("4. Extracted Declarations",styles["Heading2"])]
    drows=[["Field","Value","Status","Confidence","Sources"]]
    for f,d in result["declarations"].items():
        drows.append([f,str(d.get("value") or "")[:80],d.get("status",""),
                      f"{d.get('confidence',0):.1f}",", ".join(d.get("source_images",[]))])
    dt=Table(drows,colWidths=[38*mm,55*mm,30*mm,25*mm,35*mm],repeatRows=1)
    dt.setStyle(TableStyle([("GRID",(0,0),(-1,-1),.3,colors.grey),
                            ("BACKGROUND",(0,0),(-1,0),colors.whitesmoke),
                            ("FONTSIZE",(0,0),(-1,-1),7)]))
    story += [dt]

    story += [Paragraph("5. Multi-Image Conflicts",styles["Heading2"])]
    story += [Paragraph("None detected." if not result["conflicts"] else
                        "<br/>".join(str(c) for c in result["conflicts"]),styles["BodyText"])]

    story += [Paragraph("6. Legal Findings",styles["Heading2"])]
    frows=[["Rule","Provision","Status","Detail"]]+[
        [f["rule_id"],f["provision"],f["status"],f["detail"][:130]]
        for f in result["legal_findings"]]
    ft=Table(frows,colWidths=[30*mm,35*mm,38*mm,80*mm],repeatRows=1)
    ft.setStyle(TableStyle([("GRID",(0,0),(-1,-1),.3,colors.grey),
                            ("BACKGROUND",(0,0),(-1,0),colors.whitesmoke),
                            ("FONTSIZE",(0,0),(-1,-1),7)]))
    story += [ft]

    story += [Paragraph("7. Data Quality Warnings",styles["Heading2"])]
    story += [Paragraph("<br/>".join(str(w) for w in result["data_quality_warnings"])
                        if result["data_quality_warnings"] else "None.",styles["BodyText"])]

    story += [Paragraph("8. Exemptions Applied",styles["Heading2"])]
    story += [Paragraph("<br/>".join(result["exemptions_applied"])
                        if result["exemptions_applied"] else "None automatically applied.",styles["BodyText"])]

    story += [PageBreak(),Paragraph("9. Image Evidence",styles["Heading2"])]
    for role,item in result["package_images"].items():
        p="/content/"+Path(item["filename"]).name
        if Path(p).exists():
            story += [Paragraph(f"<b>{role.upper()}</b> — {item['filename']}",styles["BodyText"]),
                      RLImage(p,width=80*mm,height=55*mm),Spacer(1,5)]

    story += [Paragraph("10. Legal Source",styles["Heading2"]),Paragraph(LEGAL_SOURCE,styles["BodyText"]),
              Paragraph("11. Legal Safety Statement",styles["Heading2"]),Paragraph(LEGAL_SAFETY,styles["BodyText"]),
              Paragraph("12. Physical Quantity Limitation",styles["Heading2"]),Paragraph(QUANTITY_LIMIT,styles["BodyText"])]
    doc.build(story)
    return output_path

inspection_result=build_result()
report_path=generate_report(inspection_result,
    f"/content/NiriKsha_Inspection_Report_{inspection_input['inspection_id']}.pdf")
print("Report:",report_path)

In [ ]:
from IPython.display import FileLink, display
print("=== FINAL SUMMARY ===")
print("Status:", inspection_result["overall_status"])
print("Views:", list(package_images))
print("Conflicts:", len(conflicts))
print("Legal findings:", len(inspection_result["legal_findings"]))
print("Data-quality warnings:", len(inspection_result["data_quality_warnings"]))
display(FileLink(report_path))

# 8. Standalone combined function for FastAPI porting

The reusable pipeline below accepts normal Python data and images. Colab upload/display calls are not inside it.

In [ ]:
def run_inspection(product_info, images, output_pdf="/content/NiriKsha_Inspection_Report.pdf"):
    quality, failed = run_quality_gate(images)
    if failed:
        return {"overall_status":"RETAKE_REQUIRED","failed_images":failed,
                "image_quality":quality,
                "message":"Retake the blurry image(s) before continuing."}
    ocr=run_ocr(images,quality)
    per_image={r:extract_declarations(o) for r,o in ocr.items()}
    merged, conflicts=merge_declarations(per_image)
    merged["commodity_name"]={
        "field":"commodity_name","value":product_info.get("product_name"),
        "status":"manual_context" if product_info.get("product_name") else "not_found",
        "confidence":100.0 if product_info.get("product_name") else 0.0,
        "source_images":[]}
    comp=check_compliance(merged,product_info)
    result={"inspection":product_info,"package_images":images,"image_quality":quality,
            "ocr":ocr,"declarations":merged,"conflicts":conflicts,
            "legal_findings":comp["legal_findings"],
            "data_quality_warnings":comp["data_quality_warnings"],
            "exemptions_applied":comp["exemptions_applied"],
            "overall_status":comp["overall_status"],
            "legal_source":LEGAL_SOURCE,"rule_version":RULE_VERSION,
            "disclaimer":LEGAL_SAFETY,"physical_quantity_disclaimer":QUANTITY_LIMIT}
    result["report_path"]=generate_report(result,output_pdf)
    return result

print("run_inspection(product_info, images) is ready.")

# 9. Prototype self-tests

These tests verify the reusable logic without needing an external API or database.

In [ ]:
def synthetic_image(sharp=True,w=800,h=600):
    img=np.zeros((h,w,3),dtype=np.uint8)
    cv2.putText(img,"MRP Rs. 250",(60,180),cv2.FONT_HERSHEY_SIMPLEX,2,(255,255,255),4)
    cv2.rectangle(img,(40,90),(720,260),(255,255,255),3)
    return cv2.GaussianBlur(img,(31,31),0) if not sharp else img

assert isinstance(check_blur(synthetic_image())["score"],float)
assert isinstance(check_blur(synthetic_image(w=200,h=150))["score"],float)

demo={
 "front":[{"field":"mrp","value":"250","status":"detected","confidence":90,
           "source_image":"front","evidence_text":"MRP 250"}],
 "back":[{"field":"mrp","value":"260","status":"detected","confidence":91,
          "source_image":"back","evidence_text":"MRP 260"}]
}
m,c=merge_declarations(demo)
assert m["mrp"]["status"]=="conflicting" and len(c)==1

missing=check_compliance({"mrp":{"value":None,"status":"not_found","confidence":0,"source_images":[]}},
                         {"imported":False})
assert any(x["status"]=="NEEDS_MANUAL_VERIFICATION" for x in missing["legal_findings"])

json.dumps({"status":missing["overall_status"],"findings":missing["legal_findings"]})
print("All prototype self-tests passed.")

# Limitations and backend handoff

- The legal engine is intentionally a **focused subset**, not every provision in the supplied Rules.
- OCR non-detection is not treated as automatic non-compliance.
- Conflicting values require manual verification.
- Physical quantity cannot be verified from a photograph.
- Exact statutory font-size certification is not attempted.
- Rule 26 exemptions that require facts not safely inferable from the image are not guessed.
- Before production use, each enabled legal rule should be independently reviewed against the responsible legal/QA source.

Reusable backend functions:
`check_blur`, `preprocess_for_ocr`, `extract_text`, `extract_declarations`, `merge_declarations`, `check_compliance`, `generate_report`, `run_inspection`.